# Full experiment suite (Kaggle GPU)

Runs the repeated-seed experiments, paired bootstrap tests, and evaluation
analysis that require a GPU or the satellite patches.

**Setup (once per session):**
1. Right panel -> *Add Input* -> attach **both** datasets:
   `mesogeos-track-a-processed` (weather/tabular) and `mesogeos-patches` (satellite windows).
2. Right panel -> *Session options* -> **Accelerator = GPU T4**.
   Do **not** use the P100: it is compute capability sm_60 and the preinstalled
   PyTorch build supports sm_70+, so kernels fail to launch.
3. Run all cells top to bottom.

Outputs land in `/kaggle/working/runs/seeds` and `/kaggle/working/figures`;
download them and copy the CSVs into the dissertation.

In [ ]:
%cd /kaggle/working

In [ ]:
!rm -rf wildfire-dissertation
!git clone -q https://github.com/Aaffnnaann/wildfire-dissertation.git
!pip -q install scikit-learn

In [ ]:
REPO = '/kaggle/working/wildfire-dissertation'
RUNS = '/kaggle/working/runs/seeds'
FIGS = '/kaggle/working/figures'
SEEDS = '0 1 2 3 4'
import torch, subprocess
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE')
print('commit:', subprocess.run(['git','-C',REPO,'rev-parse','--short','HEAD'],
                                capture_output=True, text=True).stdout.strip())

## 1. Repeated-seed runs

Weather-only and classical models first (fast), then the satellite and fusion
models (slower; these need the patches dataset attached).

In [ ]:
!cd {REPO} && python -m wildfire.multiseed \
    --models hist_grad_boost gru lstm cnn1d bilstm rnn temporal_transformer gtn \
    --seeds {SEEDS} --out_dir {RUNS} --ensemble

In [ ]:
!cd {REPO} && python -m wildfire.multiseed \
    --models vit_v2 fusion_concat fusion_cross_v2 \
    --seeds {SEEDS} --out_dir {RUNS} --ensemble

In [ ]:
# aggregate everything, recomputing the 4-member ensemble per seed
!cd {REPO} && python -m wildfire.multiseed --collect_only \
    --models hist_grad_boost random_forest gru lstm cnn1d bilstm rnn \
             temporal_transformer gtn vit_v2 fusion_concat fusion_cross_v2 \
    --seeds {SEEDS} --out_dir {RUNS} \
    --ensemble hist_grad_boost gru cnn1d fusion_cross_v2

## 2. Paired bootstrap tests

Confidence intervals for AUPRC differences on the fixed test set.

In [ ]:
!cd {REPO} && python -m wildfire.stats_tests --runs {RUNS} --n_boot 10000 \
    --pairs hist_grad_boost:gru gru:fusion_cross_v2 hist_grad_boost:ensemble \
            temporal_transformer:gtn fusion_concat:fusion_cross_v2

## 3. Evaluation analysis

PR curves, reliability curves, Brier/ECE, validation-selected confusion
matrices, and error breakdowns by region, season and burned-area band.

In [ ]:
!cd {REPO} && python -m wildfire.evaluation --runs {RUNS} --out_dir {FIGS} \
    --models hist_grad_boost gru temporal_transformer fusion_cross_v2 ensemble \
    --error_model ensemble

## 4. Count-matched sensitivity check

Rebuilds the dataset with the paper's exact per-split totals (194 negatives
dropped) and re-runs the two cheapest strong models, to test whether the extra
negatives affect any conclusion.

In [ ]:
!cd {REPO} && python -m wildfire.paper_matched --seed 0 \
    --src $(python -c "from wildfire.train import resolve_data_root as r; print(r('x'))") \
    --dst /kaggle/working/processed_paper_matched
!cd {REPO} && python -m wildfire.multiseed --models hist_grad_boost gru \
    --seeds {SEEDS} --out_dir /kaggle/working/runs/paper_matched --ensemble

## 5. Results summary

In [ ]:
import pandas as pd, pathlib
for name in ['seed_summary.csv', 'per_seed_metrics.csv', 'bootstrap_tests.csv']:
    p = pathlib.Path(RUNS) / name
    if p.exists():
        print('\n===', name, '===')
        display(pd.read_csv(p))
for name in ['calibration_metrics.csv', 'confusion_matrices.csv',
             'error_by_region.csv', 'error_by_season.csv', 'error_by_burned_area.csv']:
    p = pathlib.Path(FIGS) / name
    if p.exists():
        print('\n===', name, '===')
        display(pd.read_csv(p))